# Predator-prey: sparse-learning-CRN LASSO, black-box and mass-action versions

This notebook produces the two LASSO columns of the predator-prey table (example 4, $T=40$, $\lambda=0.01$) with the
sparse-learning-CRN code of Zhang et al. (https://github.com/zwpku/sparse-learning-CRN), **without modifying their code**,
and writes `writing/table_predprey_detail_massaction.tex`.

| | Black-box version | Mass-action version |
|---|---|---|
| Run folder | `sparse_runs/example4_T40_lam0p01_blackbox` | `sparse_runs/example4_T40_lam0p01_massaction` |
| Basis functions per channel | all 6: $1, A, B, A^2, AB, B^2$ (their default) | only the basis functions of our candidate reactions in that channel: $1, x, x(x-1), xy$ |
| `xx_basis_flag` | 1 (quadratic basis $x^2$, their default) | 0 (quadratic basis $x(x-1)$, i.e. mass action) |
| Reported as | polynomial propensity | rate constant of each candidate reaction ($k = 2c$ for $2X$ reactants, to match $k\,X(X-1)/2$) |

Everything else is identical: same trajectory, same $\lambda$, same solver (FISTA) and the **same stopping settings**.

### Stopping settings
Their default stopping rule (stop when the cost has not improved for 1,000 steps) ended some channels before
convergence; in the original black-box run this changed the $(-1,0)$ channel from about $0.026 + 0.080A + 0.016B$
(the actual LASSO solution) to $0.018A + 0.036B$. We therefore run both versions with a longer budget
(`tot_step = 300000`, `max_step_since_prev_min_cost = 100000`, `cost_stop_tol = 1e-12`), and **section 7 checks every
channel independently**: it re-implements their objective in Python, confirms it reproduces the cost in their log, minimises
it with a different optimiser, and reports the gap.

### What their code does not enforce
The mass-action version cannot force rates to be non-negative (their proximal step is an ordinary soft-threshold), so small
negative values can appear for absent reactions. They are flagged.

### Two modes
* **`RUN_LASSO = False` (default): post-processing only.** Reads the saved outputs of both runs, the Bayesian summary
  `results/example4_T40/mcmc_summary.xlsx`, and rebuilds the tables. Needs no compiled programs.
* **`RUN_LASSO = True`: re-runs both LASSO fits** with their compiled programs (`prepare`, `sparse_learning`), then rebuilds
  the tables. The black-box fit is slow (about 30 minutes on our test machine, because its hardest channel uses the full 300,000 steps); the mass-action fit takes about 2 minutes.

In neither mode is the trajectory re-simulated (it is read from `data/example4_T40_trajectory.json`), and the Bayesian MCMC
is not re-run (that is `example4.ipynb`).


## 1. Settings

In [1]:
import os, sys, re, shutil, subprocess
import numpy as np
import pandas as pd
from scipy.optimize import minimize

REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from src.parsing import export_for_sparse_learning, load_reaction_network

# False: post-processing only (read saved outputs, rebuild tables).
# True:  re-run both LASSO fits with their compiled programs, then rebuild tables.
RUN_LASSO = False
OVERWRITE = False          # only used when RUN_LASSO = True: allow replacing existing run folders

# Location of the compiled sparse-learning-CRN programs (only needed when RUN_LASSO = True).
SPARSE_LEARNING_SRC = os.environ.get(
    "SPARSE_LEARNING_SRC",
    "/Users/suzannesindi/Dropbox/WritingProjects/2026_06_08_CRNWork/sparse-learning-CRN/src",
)
PREPARE_EXE = os.path.join(SPARSE_LEARNING_SRC, "prepare")
SPARSE_EXE  = os.path.join(SPARSE_LEARNING_SRC, "sparse_learning")

NETWORK_FILE = os.path.join(REPO, "data", "example4_network.json")
TRAJ_FILE    = os.path.join(REPO, "data", "example4_T40_trajectory.json")
MCMC_FILE    = os.path.join(REPO, "results", "example4_T40", "mcmc_summary.xlsx")
TEX_OUT      = os.path.join(REPO, "writing", "table_predprey_detail_massaction.tex")

T      = 40
LAMBDA = 0.01
STOP   = {"tot_step": 300000, "max_step_since_prev_min_cost": 100000, "cost_stop_tol": 1e-12}

lam_str = str(LAMBDA).replace(".", "p")
RUN_DIRS = {
    "blackbox":   os.path.join(REPO, "sparse_runs", f"example4_T{T}_lam{lam_str}_blackbox"),
    "massaction": os.path.join(REPO, "sparse_runs", f"example4_T{T}_lam{lam_str}_massaction"),
}

needed = [NETWORK_FILE, MCMC_FILE] + ([PREPARE_EXE, SPARSE_EXE, TRAJ_FILE] if RUN_LASSO else list(RUN_DIRS.values()))
for p in needed:
    print(("ok      " if os.path.exists(p) else "MISSING ") + p)
print("\nMode:", "re-run both LASSO fits" if RUN_LASSO else "post-processing only")
for name, d in RUN_DIRS.items():
    if RUN_LASSO and os.path.exists(d) and not OVERWRITE:
        raise RuntimeError(f"{d} already exists. Set OVERWRITE = True to replace it, or RUN_LASSO = False.")
    if not RUN_LASSO and not os.path.exists(d):
        raise RuntimeError(f"{d} not found. Set RUN_LASSO = True to create it.")


ok      /home/claude/w/BayesCRNInference/data/example4_network.json
ok      /home/claude/w/BayesCRNInference/results/example4_T40/mcmc_summary.xlsx
ok      /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_blackbox
ok      /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction

Mode: post-processing only


## 2. Candidate mass-action reactions per channel

Taken from the same reaction library as the Bayesian analysis (`compatible_reactions` in `example4_network.json`).

In [2]:
(rn4, _, rnames4, pnames4, trueTheta4, pvals4, sampled4,
 rmat4, uc4, cr4, sp4) = load_reaction_network(NETWORK_FILE)

true_k = {gi: pvals4[p] for gi, p in zip(sampled4, pnames4)}   # reaction index -> true rate

def side(v):
    parts = [s if int(n) == 1 else f"{int(n)}{s}" for n, s in zip(v, sp4) if int(n) != 0]
    return "+".join(parts) if parts else "∅"

def rxn_str(gi, change):
    reac = rmat4[:, gi]
    return f"{side(reac)} -> {side(reac + np.array(change))}"

CHANNELS = [(1, 0), (-1, 0), (-1, 1), (0, -1)]    # channels observed in the trajectory, in table order
for ch in CHANNELS:
    print(f"channel {ch}:")
    for gi in cr4[ch]:
        print(f"    {rxn_str(gi, ch):14s}  true k = {true_k.get(gi, 0.0)}")


Reaction network loaded from /home/claude/w/BayesCRNInference/data/example4_network.json
  Species:   ['A', 'B']
  Reactions: 4 sampled  |  30 total in full CRN
  Unique stoichiometric changes: 18
channel (1, 0):
    ∅ -> A          true k = 0.0
    A -> 2A         true k = 1.0
    B -> A+B        true k = 0.0
channel (-1, 0):
    A -> ∅          true k = 0.1
    2A -> A         true k = 0.0
    A+B -> B        true k = 0.0
channel (-1, 1):
    A -> B          true k = 0.0
    2A -> A+B       true k = 0.0
    A+B -> 2B       true k = 0.01
channel (0, -1):
    B -> ∅          true k = 0.5
    2B -> B         true k = 0.0
    A+B -> A        true k = 0.0


## 3. Helper functions

Reading and writing their file formats. `basis_of_channel_<i>.txt` holds: the number of basis functions, their indices,
a weight per basis function, an initial value per basis function, then the minimum and maximum of each basis function along
the trajectory.

In [3]:
def run(exe, cwd):
    print(f"$ cd {cwd}\n$ {exe}")
    p = subprocess.run([exe], cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:], p.stderr)
        raise RuntimeError(f"{os.path.basename(exe)} exited with code {p.returncode}")
    print(f"[{os.path.basename(exe)} finished, exit code 0]")

def read_channel_info(out_dir):
    tok = open(os.path.join(out_dir, "channel_info.txt")).read().split()
    n_ch, n_sp = int(tok[0]), int(tok[1])
    vals = list(map(int, tok[2:]))
    changes = [tuple(vals[i*n_sp:(i+1)*n_sp]) for i in range(n_ch)]
    counts = vals[n_ch*n_sp:n_ch*n_sp + n_ch]
    return changes, counts

def read_basis_info(out_dir):
    """Basis functions as tuples of species indices: () = 1, (0,) = A, (0,1) = AB, (0,0) = A^2 or A(A-1)."""
    tok = list(map(int, open(os.path.join(out_dir, "basis_funct_info.txt")).read().split()))
    basis, pos = [], 2
    for _ in range(tok[1]):
        k = tok[pos]; basis.append(tuple(tok[pos+1:pos+1+k])); pos += 1 + k
    return basis

def read_basis_file(path):
    tok = open(path).read().split()
    n = int(tok[0]); v = tok[1:]
    return n, [v[k*n:(k+1)*n] for k in range(len(v) // n)]

def read_omega(out_dir, ch_i):
    tok = open(os.path.join(out_dir, f"omega_vec_for_channel_{ch_i}.txt")).read().split()
    return np.array(list(map(float, tok[1:1 + int(tok[0])])))

def cfg_value(run_dir, key):
    cfg = open(os.path.join(run_dir, "sparse_learning.cfg")).read()
    return float(re.search(rf"^{key} = ([^ ;]+) ;", cfg, re.M).group(1))

def basis_label(b, xx_flag):
    if len(b) == 0: return "1"
    if len(b) == 1: return sp4[b[0]]
    i, j = b
    if i == j: return f"{sp4[i]}({sp4[i]}-1)" if xx_flag == 0 else f"{sp4[i]}^2"
    return sp4[i] + sp4[j]

def reactant_key(reac):
    return tuple(sorted(s for s, n in enumerate(reac) for _ in range(int(n))))

def export_run(run_dir, xx_flag):
    """Export the trajectory with export_for_sparse_learning, then set xx_basis_flag and the stopping settings."""
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir)
    export_for_sparse_learning(trajectory_files=[TRAJ_FILE], working_dir=run_dir, T=T, poly_order=2,
                               regular_lambda=LAMBDA, tot_step=STOP["tot_step"], cost_stop_tol=STOP["cost_stop_tol"])
    path = os.path.join(run_dir, "sparse_learning.cfg")
    cfg = open(path).read()
    cfg = re.sub(r"^xx_basis_flag = \d ;$", f"xx_basis_flag = {xx_flag} ;", cfg, flags=re.M)
    cfg = re.sub(r"^max_step_since_prev_min_cost = \d+ ;$",
                 f"max_step_since_prev_min_cost = {STOP['max_step_since_prev_min_cost']} ;", cfg, flags=re.M)
    open(path, "w").write(cfg)

def show_cfg(run_dir):
    cfg = open(os.path.join(run_dir, "sparse_learning.cfg")).read()
    keys = ["T", "regular_lambda", "poly_order", "xx_basis_flag", "solver_id", "tot_step",
            "max_step_since_prev_min_cost", "cost_stop_tol"]
    print("   " + ",  ".join(f"{k} = {cfg_value(run_dir, k):g}" for k in keys))


## 4. Black-box version

Their default set-up: every channel uses all six basis functions $1, A, B, A^2, AB, B^2$ with `xx_basis_flag = 1`.

In [4]:
BB = RUN_DIRS["blackbox"]
if RUN_LASSO:
    export_run(BB, xx_flag=1)
    run(PREPARE_EXE, BB)
    run(SPARSE_EXE, BB)          # slow: about 30 minutes
else:
    print("RUN_LASSO = False: reading saved outputs in", BB)
show_cfg(BB)
assert cfg_value(BB, "xx_basis_flag") == 1


RUN_LASSO = False: reading saved outputs in /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_blackbox
   T = 40,  regular_lambda = 0.01,  poly_order = 2,  xx_basis_flag = 1,  solver_id = 1,  tot_step = 300000,  max_step_since_prev_min_cost = 100000,  cost_stop_tol = 1e-12


## 5. Mass-action version

Same export, but `xx_basis_flag = 0`, and after `prepare` each channel's basis file is cut down to the basis functions of
our candidate reactions (the step their instructions describe, `examples/HowToRun`, step 3). The original files are kept as
`basis_of_channel_<i>_full.txt`.

In [5]:
MA = RUN_DIRS["massaction"]
if RUN_LASSO:
    export_run(MA, xx_flag=0)
    run(PREPARE_EXE, MA)
else:
    print("RUN_LASSO = False: reading saved outputs in", MA)
show_cfg(MA)
assert cfg_value(MA, "xx_basis_flag") == 0

OUT_MA = os.path.join(MA, "output")
ma_changes, ma_counts = read_channel_info(OUT_MA)
ma_basis = read_basis_info(OUT_MA)
assert sorted(ma_changes) == sorted(CHANNELS), "unexpected channels"

KEEP = {}   # channel number -> list of (basis index, reaction index)
for ch_i, change in enumerate(ma_changes):
    path = os.path.join(OUT_MA, f"basis_of_channel_{ch_i}.txt")
    full = path.replace(".txt", "_full.txt")
    keep = [(ma_basis.index(reactant_key(rmat4[:, gi])), gi) for gi in cr4[change]]
    KEEP[ch_i] = keep
    if RUN_LASSO:
        shutil.copy2(path, full)
        n, fields = read_basis_file(full)
        idx_full = [int(x) for x in fields[0]]
        pos = [idx_full.index(b) for b, _ in keep]
        with open(path, "w") as f:
            f.write(f"{len(pos)} \n")
            for k, row in enumerate(fields):
                f.write(" ".join(row[p] for p in pos) + " \n")
                if k == 2:
                    f.write("\n")
    _, saved = read_basis_file(path)
    assert [int(x) for x in saved[0]] == [b for b, _ in keep], f"basis file of channel {ch_i} does not match"
    print(f"channel {ch_i} {change}: basis {[basis_label(ma_basis[b], 0) for b, _ in keep]}"
          f"  <->  {[rxn_str(gi, change) for _, gi in keep]}")

if RUN_LASSO:
    run(SPARSE_EXE, MA)          # about 2 minutes


RUN_LASSO = False: reading saved outputs in /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction
   T = 40,  regular_lambda = 0.01,  poly_order = 2,  xx_basis_flag = 0,  solver_id = 1,  tot_step = 300000,  max_step_since_prev_min_cost = 100000,  cost_stop_tol = 1e-12
channel 0 (-1, 0): basis ['A', 'A(A-1)', 'AB']  <->  ['A -> ∅', '2A -> A', 'A+B -> B']
channel 1 (-1, 1): basis ['A', 'A(A-1)', 'AB']  <->  ['A -> B', '2A -> A+B', 'A+B -> 2B']
channel 2 (0, -1): basis ['B', 'B(B-1)', 'AB']  <->  ['B -> ∅', '2B -> B', 'A+B -> A']
channel 3 (1, 0): basis ['1', 'A', 'B']  <->  ['∅ -> A', 'A -> 2A', 'B -> A+B']


## 6. How each channel's fit stopped (from their log)

"Changes of the costs ... under threshold" means the fit converged. "Maximal steps" or "Minimal cost hasn't been updated"
means it stopped for another reason and reports the lowest-cost point it found; section 7 checks how far that is from the
optimum.

In [6]:
def termination_summary(run_dir, changes):
    log_path = os.path.join(run_dir, "log", "sparse_infer.log")
    if not os.path.exists(log_path):
        print(f"(no log in {os.path.relpath(run_dir, REPO)}: logs are not tracked by git)")
        return None
    log = open(log_path).read()
    blocks = re.split(r"Solving coefficients for the reaction channel (\d+)\.\.\. finished\.", log)
    rows = []
    for k in range(1, len(blocks), 2):
        ch, body = int(blocks[k]), blocks[k+1]
        reason = re.search(r"Reason of termination:\s*(.*)", body)
        steps  = re.search(r"Total steps=(\d+),.*?min_cost=(\S+) \(at Step (\d+)\)", body)
        rows.append({"channel": changes[ch], "reason": reason.group(1).strip() if reason else "?",
                     "steps": int(steps.group(1)) if steps else None,
                     "best cost": float(steps.group(2)) if steps else None})
    return pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 80)
for name, d in RUN_DIRS.items():
    print(f"\n{name}:")
    df = termination_summary(d, read_channel_info(os.path.join(d, "output"))[0])
    if df is not None:
        display(df)



blackbox:


,channel,reason,steps,best cost
0,"(-1, 0)",Maximal steps (300000) achieved.,300000,-4.330985
1,"(-1, 1)",Changes of the costs in the last 20 steps are under threshold.,84154,-161.654902
2,"(0, -1)",Changes of the costs in the last 20 steps are under threshold.,51251,-141.584905
3,"(1, 0)",Minimal cost hasn't been updated after 100000 steps.,106870,-161.183863



massaction:


,channel,reason,steps,best cost
0,"(-1, 0)",Changes of the costs in the last 20 steps are under threshold.,23559,-4.256057
1,"(-1, 1)",Changes of the costs in the last 20 steps are under threshold.,3219,-161.643798
2,"(0, -1)",Changes of the costs in the last 20 steps are under threshold.,32782,-141.579436
3,"(1, 0)",Changes of the costs in the last 20 steps are under threshold.,13613,-161.175276


## 7. Independent check of every channel

For each channel we re-implement their objective (from `utils.cpp`):

$$
f(c) = \frac{1}{T_{\mathrm{obs}}}\Big[-\sum_{\text{jumps } n \text{ in channel}} \log G\big(a_c(x_n)\big)
+ \sum_{\text{states } i} \tau_i\, G\big(a_c(x_i)\big)\Big] + \lambda \sum_j |c_j|,
\qquad a_c(x) = \sum_j c_j \phi_j(x),
$$

where $G(x) = \varepsilon \log(1 + e^{x/\varepsilon})$ is their smoothed positive part ($\varepsilon$ = `eps` in the
config), and $T_{\mathrm{obs}}$ is the total holding time in the trajectory file (their `total_T`, the time of the last event).
We then:
1. evaluate $f$ at their saved coefficients and compare with the cost in their log (if the log is present);
2. minimise $f$ ourselves (L-BFGS-B on $c = u - v$, $u, v \ge 0$, several starting points);
3. report the gap $f(\text{theirs}) - f(\text{ours})$.

A gap near zero means their result is the LASSO solution. When a channel's objective is very flat, very different small
coefficients can have nearly the same cost; only the dominant terms are then well determined.

In [7]:
def check_run(run_dir):
    eps, gcut = cfg_value(run_dir, "eps"), cfg_value(run_dir, "g_cut")
    lam, xx = cfg_value(run_dir, "regular_lambda"), int(cfg_value(run_dir, "xx_basis_flag"))
    d = np.loadtxt(os.path.join(run_dir, "traj_data", "traj_0.txt"), skiprows=1)
    X, tau = d[:, 1:-1], d[:, -1]
    T_obs = tau.sum()
    out = os.path.join(run_dir, "output")
    basis = read_basis_info(out)
    def phi(b):
        if len(b) == 0: return np.ones(len(X))
        if len(b) == 1: return X[:, b[0]]
        i, j = b
        return X[:, i] * (X[:, i] - 1) if (i == j and xx == 0) else X[:, i] * X[:, j]
    PHI = np.stack([phi(b) for b in basis], 1)
    G = lambda x: np.where(x/eps >= gcut, x, eps*np.log1p(np.exp(np.minimum(x/eps, gcut))))
    def logG(x):
        t = x/eps
        with np.errstate(divide="ignore", invalid="ignore"):
            small = np.log(np.log1p(np.exp(np.minimum(t, gcut)))) + np.log(eps)
            small = np.where(np.isinf(small), t + np.log(eps), small)
            big = np.log(np.abs(x)) + np.log1p(np.log1p(np.exp(-np.abs(t)))/np.abs(t))
        return np.where(t > gcut, big, small)
    logcost = {}
    log_path = os.path.join(run_dir, "log", "sparse_infer.log")
    if os.path.exists(log_path):
        for m in re.finditer(r"channel (\d+)\.\.\. finished\..*?Corresponding cost : (\S+)", open(log_path).read(), re.S):
            logcost[int(m.group(1))] = float(m.group(2))
    changes, _ = read_channel_info(out)
    dX = np.diff(X, axis=0)
    rows, ours = [], {}
    for i, change in enumerate(changes):
        n, fields = read_basis_file(os.path.join(out, f"basis_of_channel_{i}.txt"))
        idx = [int(x) for x in fields[0]]
        P = PHI[:, idx]
        jump = np.r_[np.all(dX == np.array(change), axis=1), False]
        f = lambda c: (-logG(P[jump] @ c).sum() + (tau * G(P @ c)).sum()) / T_obs + lam * np.abs(c).sum()
        theirs = read_omega(out, i)
        sc = np.maximum(np.abs(P).mean(0), 1e-12)
        fs = lambda z: f(z[:n]/sc - z[n:]/sc)
        starts = [np.r_[np.maximum(theirs, 0), np.maximum(-theirs, 0)] * np.r_[sc, sc], np.full(2*n, 0.01)]
        starts += [np.random.default_rng(s).uniform(0, 2, 2*n) for s in range(4)]
        best = min((minimize(fs, s0, method="L-BFGS-B", bounds=[(0, None)]*(2*n),
                             options=dict(maxiter=20000, maxfun=200000, ftol=1e-15, gtol=1e-12)) for s0 in starts),
                   key=lambda r: r.fun)
        c_ours = best.x[:n]/sc - best.x[n:]/sc
        ours[change] = c_ours
        rows.append({"channel": change,
                     "their cost (log)": logcost.get(i, np.nan),
                     "f(their coeffs)": f(theirs),
                     "f(our optimum)": best.fun,
                     "gap": f(theirs) - best.fun,
                     "basis": " ".join(basis_label(basis[b], xx) for b in idx),
                     "their coeffs": " ".join(f"{v:+.4f}" for v in theirs),
                     "our coeffs": " ".join(f"{v:+.4f}" for v in c_ours)})
    return pd.DataFrame(rows)

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
checks = {}
for name, d in RUN_DIRS.items():
    print(f"\n{name}:")
    checks[name] = check_run(d)
    display(checks[name])
for name, df in checks.items():
    worst = df["gap"].max()
    print(f"{name}: largest gap = {worst:.2e}  ->  " +
          ("all channels at the LASSO optimum (to within 1e-3)" if worst < 1e-3 else "CHECK: a channel is not at the optimum"))



blackbox:


,channel,their cost (log),f(their coeffs),f(our optimum),gap,basis,their coeffs,our coeffs
0,"(-1, 0)",-4.330985,-4.330986,-4.330986,0.000000,1 A B A^2 AB B^2,+0.0256 +0.0805 +0.0159 -0.0000 +0.0001 -0.0001,+0.0260 +0.0805 +0.0159 -0.0000 +0.0001 -0.0001
1,"(-1, 1)",-161.654900,-161.654943,-161.654943,0.000000,1 A B A^2 AB B^2,+0.8586 -0.0236 -0.0086 +0.0001 +0.0102 -0.0000,+0.8662 -0.0238 -0.0087 +0.0001 +0.0102 -0.0000
2,"(0, -1)",-141.584900,-141.584940,-141.584940,0.000000,1 A B A^2 AB B^2,-0.0000 -0.0256 +0.4916 +0.0001 -0.0001 +0.0001,-0.0000 -0.0257 +0.4917 +0.0001 -0.0001 +0.0001
3,"(1, 0)",-161.183900,-161.183903,-161.184004,0.000102,1 A B A^2 AB B^2,+0.0249 +0.9697 +0.0020 +0.0000 +0.0004 -0.0000,+0.0001 +0.9683 +0.0015 +0.0001 +0.0004 -0.0000



massaction:


,channel,their cost (log),f(their coeffs),f(our optimum),gap,basis,their coeffs,our coeffs
0,"(-1, 0)",-4.256057,-4.256058,-4.256058,0.000000,A A(A-1) AB,+0.1096 -0.0001 +0.0000,+0.1096 -0.0001 +0.0000
1,"(-1, 1)",-161.643800,-161.643838,-161.643838,0.000000,A A(A-1) AB,+0.0029 +0.0001 +0.0100,+0.0029 +0.0001 +0.0100
2,"(0, -1)",-141.579400,-141.579472,-141.579472,0.000000,B B(B-1) AB,+0.4782 +0.0002 -0.0002,+0.4782 +0.0002 -0.0002
3,"(1, 0)",-161.175300,-161.175316,-161.175316,0.000000,1 A B,+0.0000 +1.0041 -0.0013,+0.0000 +1.0041 -0.0013


blackbox: largest gap = 1.02e-04  ->  all channels at the LASSO optimum (to within 1e-3)
massaction: largest gap = 2.84e-14  ->  all channels at the LASSO optimum (to within 1e-3)


## 8. Comparison table

LASSO coefficients are read from the run folders; rates use the Bayesian convention $k = 2c$ for $2X$ reactants.

In [8]:
def rate_factor(reac):
    return 2.0 if any(int(n) == 2 for n in reac) else 1.0

df4 = pd.read_excel(MCMC_FILE)
count_to_ch = {n: c for c, n in zip(ma_changes, ma_counts)}
run_to_ch = {int(ri): count_to_ch.get(int(g["Count"].iloc[0])) for ri, g in df4.groupby("Run_Index")}
mcmc = {(run_to_ch[int(r["Run_Index"])], int(r["Param_Index"])): r
        for _, r in df4.iterrows() if run_to_ch.get(int(r["Run_Index"])) is not None}

OUT_BB = os.path.join(BB, "output")
bb_changes, _ = read_channel_info(OUT_BB)
bb_basis = read_basis_info(OUT_BB)
bb_idx = {ch: [int(x) for x in read_basis_file(os.path.join(OUT_BB, f"basis_of_channel_{i}.txt"))[1][0]]
          for i, ch in enumerate(bb_changes)}
bb_coef = {ch: read_omega(OUT_BB, i) for i, ch in enumerate(bb_changes)}

def bb_terms(change, thr=1e-3):
    """Black-box polynomial: terms with |c| >= thr, and whether any term is impossible under mass action."""
    allowed = {reactant_key(rmat4[:, gi]) for gi in cr4[change]}
    terms, bad = [], False
    for b, v in zip(bb_idx[change], bb_coef[change]):
        if abs(v) < thr: continue
        if ma_basis_key(bb_basis[b]) not in allowed or v < 0: bad = True
        terms.append((v, bb_basis[b]))
    return terms, bad

def ma_basis_key(b):       # basis tuple -> reactant key (same convention)
    return tuple(sorted(b))

rows = []
for ch in CHANNELS:
    ch_i = ma_changes.index(ch)
    c = read_omega(OUT_MA, ch_i)
    terms, bad = bb_terms(ch)
    for (b, gi), coef in zip(KEEP[ch_i], c):
        m = mcmc.get((ch, cr4[ch].index(gi)))
        rows.append({
            "channel": str(ch), "reaction": rxn_str(gi, ch), "true k": true_k.get(gi, 0.0),
            "Bayes mean": None if m is None else m["Mean"],
            "Bayes 95% CI": None if m is None else f"({m['CI_lower']:.4f}, {m['CI_upper']:.4f})",
            "Pr(off)": None if m is None else m["Prob_Off"],
            "LASSO mass-action k": rate_factor(rmat4[:, gi]) * coef,
            "LASSO black-box": (" ".join(f"{v:+.3f}·{basis_label(bb, 1)}" for v, bb in terms) + (" †" if bad else ""))
                               if gi == cr4[ch][0] else "",
        })
res = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(res)
res.to_csv(os.path.join(MA, "comparison_table.csv"), index=False)


,channel,reaction,true k,Bayes mean,Bayes 95% CI,Pr(off),LASSO mass-action k,LASSO black-box
0,"(1, 0)",∅ -> A,0.0000,0.0045,"(0.0000, 0.0000)",0.9947,0.0000,+0.025·1 +0.970·A +0.002·B
1,"(1, 0)",A -> 2A,1.0000,1.0024,"(0.9587, 1.0480)",0.0000,1.0041,
2,"(1, 0)",B -> A+B,0.0000,0.0000,"(0.0000, 0.0000)",1.0000,-0.0013,
3,"(-1, 0)",A -> ∅,0.1000,0.0995,"(0.0859, 0.1140)",0.0000,0.1096,+0.026·1 +0.081·A +0.016·B †
4,"(-1, 0)",2A -> A,0.0000,0.0000,"(0.0000, 0.0000)",1.0000,-0.0003,
5,"(-1, 0)",A+B -> B,0.0000,0.0000,"(0.0000, 0.0000)",1.0000,0.0000,
6,"(-1, 1)",A -> B,0.0000,0.0000,"(0.0000, 0.0000)",1.0000,0.0029,+0.859·1 -0.024·A -0.009·B +0.010·AB †
7,"(-1, 1)",2A -> A+B,0.0000,0.0000,"(0.0000, 0.0000)",1.0000,0.0001,
8,"(-1, 1)",A+B -> 2B,0.0100,0.0101,"(0.0096, 0.0105)",0.0000,0.0100,
9,"(0, -1)",B -> ∅,0.5000,0.4915,"(0.4695, 0.5136)",0.0000,0.4782,-0.026·A +0.492·B †


## 9. LaTeX table

Written to `writing/table_predprey_detail_massaction.tex`. Black-box entries list all terms with $|c| \ge 0.001$;
$^\dagger$ marks an entry with a term that no candidate mass-action reaction in that channel can produce, or a negative term.

In [9]:
def tex_side(v):
    parts = [s if int(n) == 1 else f"{int(n)}{s}" for n, s in zip(v, sp4) if int(n) != 0]
    return "+".join(parts) if parts else r"\emptyset"

def tex_rxn(gi, change):
    reac = rmat4[:, gi]
    return f"${tex_side(reac)}\\to {tex_side(reac + np.array(change))}$"

def tex_num(v, nd=4):
    s = f"{v:.{nd}f}"
    if float(s) == 0: s = s.lstrip("-")
    return f"${s}$"

def tex_bb(change):
    terms, bad = bb_terms(change)
    lab = lambda b: "" if len(b) == 0 else (sp4[b[0]] if len(b) == 1 else
                    (f"{sp4[b[0]]}^2" if b[0] == b[1] else sp4[b[0]] + sp4[b[1]]))
    s = " ".join(f"{'-' if v < 0 else '+'} {abs(v):.3f}" + (f"\\,{lab(b)}" if lab(b) else "") for v, b in terms)
    s = s[2:] if s.startswith("+ ") else "-" + s[2:]
    return f"${s}{'^{\\,\\dagger}' if bad else ''}$"

lasso_k = {(r["channel"], r["reaction"]): r["LASSO mass-action k"] for _, r in res.iterrows()}
n_events = dict(zip(ma_changes, ma_counts))
chlabel = {c: f"$({c[0]:+d},\\;{c[1]:+d})$".replace("+0", "0") for c in CHANNELS}

L = [r"% Predator-prey detail with black-box and mass-action LASSO",
     r"% Generated by examples/example4_sparse_massaction.ipynb -- do not edit by hand",
     r"\begin{table}[htbp]", r"\centering", r"\small", r"\setlength{\tabcolsep}{4pt}",
     r"\caption{Predator-prey inference at $T=" + f"{T}" + r"$ (" + f"{sum(ma_counts):,}".replace(",", "{,}") +
     r" events). All candidate mass-action reactions for each stoichiometric channel $\xi_\ell$ are shown. "
     r"$k^*$: true rate (bold if active). "
     r"$\hat{k}$: posterior mean (95\% credible interval); ``---'' when $\Pr(\text{off})>0.99$. "
     r"$\Pr(\text{off})$: spike-and-slab exclusion probability. "
     r"LASSO (mass action): sparse-learning-CRN at $\lambda=" + f"{LAMBDA}" + r"$ with each channel restricted to the "
     r"basis functions of its candidate reactions ($1, x, x(x-1), xy$), reported as rate constants "
     r"($k=2c$ for $2X$ reactants). "
     r"LASSO (black box): the same method with all basis functions $1, A, B, A^2, AB, B^2$; terms with "
     r"$|c|\ge 0.001$ shown. "
     r"$^\dagger$~contains a term not produced by any mass-action reaction in that channel, or a negative term.}",
     r"\label{tab:predprey_detail}",
     r"\begin{tabular}{l l r r l r r l}", r"\toprule",
     r"$\xi_\ell$ & Reaction & $k^*$ & Events & $\hat{k}$\;(95\% CI) & $\Pr(\text{off})$ & "
     r"\shortstack{LASSO\\(mass action)} & \shortstack{LASSO\\(black box)} \\", r"\midrule"]
for ch in CHANNELS:
    cands = cr4[ch]
    for pi, gi in enumerate(cands):
        tk = true_k.get(gi, 0.0)
        m = mcmc.get((ch, pi))
        if m is None:
            post, poff = "---", "---"
        else:
            poff = f"${m['Prob_Off']:.3f}$"
            post = "---" if m["Prob_Off"] >= 0.99 else f"${m['Mean']:.4f}\\;({m['CI_lower']:.4f},\\,{m['CI_upper']:.4f})$"
        cols = [f"\\multirow{{{len(cands)}}}{{*}}{{{chlabel[ch]}}}" if pi == 0 else "",
                tex_rxn(gi, ch),
                f"\\textbf{{{tk:.4f}}}" if tk > 0 else "$0$",
                str(n_events[ch]) if pi == 0 else "",
                post, poff,
                tex_num(lasso_k[(str(ch), rxn_str(gi, ch))]),
                tex_bb(ch) if pi == 0 else ""]
        L.append(" & ".join(cols) + r" \\")
    L.append(r"\midrule")
L[-1] = r"\bottomrule"
L += [r"\end{tabular}", r"\end{table}"]
open(TEX_OUT, "w").write("\n".join(L) + "\n")
print("Saved:", TEX_OUT, "\n")
print("\n".join(L))


Saved: /home/claude/w/BayesCRNInference/writing/table_predprey_detail_massaction.tex 

% Predator-prey detail with black-box and mass-action LASSO
% Generated by examples/example4_sparse_massaction.ipynb -- do not edit by hand
\begin{table}[htbp]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Predator-prey inference at $T=40$ (5{,}803 events). All candidate mass-action reactions for each stoichiometric channel $\xi_\ell$ are shown. $k^*$: true rate (bold if active). $\hat{k}$: posterior mean (95\% credible interval); ``---'' when $\Pr(\text{off})>0.99$. $\Pr(\text{off})$: spike-and-slab exclusion probability. LASSO (mass action): sparse-learning-CRN at $\lambda=0.01$ with each channel restricted to the basis functions of its candidate reactions ($1, x, x(x-1), xy$), reported as rate constants ($k=2c$ for $2X$ reactants). LASSO (black box): the same method with all basis functions $1, A, B, A^2, AB, B^2$; terms with $|c|\ge 0.001$ shown. $^\dagger$~contains a term not produced b